In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

required = ["DWH_DB_USER", "DWH_DB_PASSWORD"]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError("Set database connection variables before running this notebook: " + ", ".join(missing))

url = URL.create(
    "postgresql+psycopg2",
    username=os.environ["DWH_DB_USER"],
    password=os.environ["DWH_DB_PASSWORD"],
    host=os.getenv("DWH_DB_HOST", "localhost"),
    port=int(os.getenv("DWH_DB_PORT", "5432")),
    database=os.getenv("DWH_DB_NAME", "DWH"),
)
engine = create_engine(url)

schema = pd.read_sql("""
    SELECT table_name, column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'public'
    ORDER BY table_name, ordinal_position
""", engine)
print(schema.to_string())


In [ ]:
q = """
SELECT f.*,
       c."BRADIE", c."CREACT", c."KMODAMRT", c."KMODPAIE", c."KFCRED_IMF", c."KNAT_SC",
       b."TOTDETTE", b."TOTDETTEPERS",
       ed."LIBELLE" AS education, pr."LIBELLE" AS profession, ec."LIBELLE" AS etat_civil,
       ap."LIBELLE" AS activite_perso, ac."LIBELLE" AS activite_contrat,
       fc."LIBELLE" AS forme_credit, nc."LIBELLE" AS nature_credit, pe."LIBELLE" AS periodicite,
       g."LIB_GOUVERNORAT" AS gouvernorat, g."LIB_DELEGATION" AS delegation,
       d.date_complete AS date_contrat, d.annee, d.trimestre, d.mois,
       dd.date_complete AS date_debloc, dr.date_complete AS date_radiation
FROM public.fact_contract f
LEFT JOIN public.dim_contract c            ON c.contract_id = f.contract_id
LEFT JOIN public.dim_beneficiaire b        ON b."BENEFICIAIRE_ID" = f."BENEFICIAIRE_ID"
LEFT JOIN public.dim_nib_education ed      ON ed.education_id = b.education_id
LEFT JOIN public.dim_profession pr         ON pr.profession_id = b.profession_id
LEFT JOIN public.dim_etat_civil ec         ON ec.etat_civil_id = b.etat_civil_id
LEFT JOIN public.dim_activite ap           ON ap."Activite_Perso_ID" = b."Activite_Perso_ID"
LEFT JOIN public.dim_activite_contract ac  ON ac."Activite_Contrat_ID" = f."Activite_Contrat_ID"
LEFT JOIN public.dim_forme_credit fc       ON fc.forme_credit_id = f.forme_credit_id
LEFT JOIN public.dim_natures_credits nc    ON nc.nature_id = f.nature_id
LEFT JOIN public.dim_periodicite pe        ON pe.periodicite_id = f.periodicite_id
LEFT JOIN public.dim_geographique g        ON g.geo_id = f.geo_id
LEFT JOIN public.dim_date d                ON d.date_id = f.date_contract_id
LEFT JOIN public.dim_date dd               ON dd.date_id = f.date_debloc_id
LEFT JOIN public.dim_date dr               ON dr.date_id = f.date_radiation_id
"""
df = pd.read_sql(q, engine)
print(df.shape, df["contract_id"].is_unique)   # attendu : 13632 lignes, True

In [ ]:
df.info()
for col in ["BRADIE", "CREACT", "TECH", "KMODAMRT", "KMODPAIE"]:
    print(df[col].value_counts(dropna=False).head(10), "\n")
print(df["date_radiation_id"].value_counts(dropna=False).head(5))
print("part avec date_radiation :", df["date_radiation"].notna().mean().round(3))
print(df[["MINITIAL","NECH","NTXTEG","NTXINTANN","NEMPNV","NEMPEXT","TOTDETTE","CMR"]].describe().T.to_string())

In [ ]:
vides = [c for c in df.columns if df[c].isna().all()]
print("colonnes vides :", vides, "\n")

print(df["education"].value_counts().head(10), "\n")
print(df["profession"].value_counts(dropna=False).head(), "\n")
print(df["etat_civil"].value_counts(dropna=False).head(), "\n")

df["conditions_manquantes"] = df["NECH"].isna().astype(int)
print(df.groupby("annee")["conditions_manquantes"].agg(["mean", "size"]).round(3), "\n")

print(df["NEMPNV"].value_counts().sort_index().head(10), "\n")
print(df["gouvernorat"].value_counts().head(5))
print("délégations :", df["delegation"].nunique(), "\n")
print(df["activite_contrat"].value_counts(), "\n")
print(df["date_contrat"].min(), df["date_contrat"].max())

print(pd.read_sql("""
    SELECT count(*) AS n, count(profession_id) AS prof, count(etat_civil_id) AS civil,
           count(education_id) AS educ, count(geo_id) AS geo
    FROM public.dim_beneficiaire
""", engine).to_string())

In [ ]:
print(df["TECH"].value_counts(dropna=False), "\n")
print(df["forme_credit"].value_counts(), "\n")
print(df["nature_credit"].value_counts(), "\n")
print(df["periodicite"].value_counts(), "\n")
print(pd.crosstab(df["conditions_manquantes"], df["NEMPNV"].clip(upper=3)))

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

df["date_contrat"] = pd.to_datetime(df["date_contrat"])
df["date_debloc"]  = pd.to_datetime(df["date_debloc"])
df["montant_tnd"]  = df["MINITIAL"] / 1000
df["delai_debloc_j"] = (df["date_debloc"] - df["date_contrat"]).dt.days

rares = df["activite_contrat"].value_counts()[lambda s: s < 100].index
df["activite_grp"] = df["activite_contrat"].where(~df["activite_contrat"].isin(rares), "Autres")

num_cols = ["NECH", "NMOISGRACE", "NTXINTANN", "NEMPEXT", "NEMPNV",
            "delai_debloc_j", "annee", "trimestre", "mois"]
cat_cols = ["TECH", "activite_grp", "forme_credit", "nature_credit",
            "periodicite", "gouvernorat", "CREACT"]

df_ml = df[df["NECH"].notna()].copy()
print("lignes utilisées :", len(df_ml))
X, y = df_ml[num_cols + cat_cols], df_ml["montant_tnd"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])
models = {
    "Baseline (médiane)": DummyRegressor(strategy="median"),
    "Ridge": Ridge(alpha=1.0),
    "Gradient boosting": HistGradientBoostingRegressor(random_state=42),
}
for name, m in models.items():
    pipe = Pipeline([("prep", prep), ("model", m)]).fit(X_tr, y_tr)
    p = pipe.predict(X_te)
    print(f"{name:20s} MAE = {mean_absolute_error(y_te, p):8.0f} TND   R² = {r2_score(y_te, p):.3f}")

In [ ]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "4"      # supprime l'avertissement joblib
from sklearn.model_selection import cross_val_score, KFold
from sklearn.inspection import permutation_importance

cat_cols = [c for c in cat_cols if c not in ["forme_credit", "nature_credit", "periodicite"]]

def make_pipe(num, cat):
    prep = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat),
    ])
    return Pipeline([("prep", prep), ("model", HistGradientBoostingRegressor(random_state=42))])

X, y = df_ml[num_cols + cat_cols], df_ml["montant_tnd"]
cv = KFold(5, shuffle=True, random_state=42)

pipe = make_pipe(num_cols, cat_cols)
mae = -cross_val_score(pipe, X, y, cv=cv, scoring="neg_mean_absolute_error")
r2 = cross_val_score(pipe, X, y, cv=cv, scoring="r2")
print(f"CV 5 plis : MAE = {mae.mean():.0f} ± {mae.std():.0f} TND | R² = {r2.mean():.3f} ± {r2.std():.3f}")

train = df_ml["annee"] <= 2022
pipe.fit(X[train], y[train])
p = pipe.predict(X[~train])
print(f"Test temporel (2023) : MAE = {mean_absolute_error(y[~train], p):.0f} TND | R² = {r2_score(y[~train], p):.3f} | n = {(~train).sum()}")

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
pipe.fit(X_tr, y_tr)
imp = permutation_importance(pipe, X_te, y_te, scoring="neg_mean_absolute_error",
                             n_repeats=5, random_state=42)
print("\nImportance (hausse de l'erreur en TND si la variable est mélangée) :")
print(pd.Series(imp.importances_mean, index=X_te.columns).sort_values(ascending=False).round(0).to_string())

print("\nAblations (CV 5 plis, MAE en TND) :")
cond = ["NECH", "NMOISGRACE", "NTXINTANN", "TECH"]
tests = {
    "toutes les variables": (num_cols, cat_cols),
    "sans NTXINTANN": ([c for c in num_cols if c != "NTXINTANN"], cat_cols),
    "sans conditions du crédit": ([c for c in num_cols if c not in cond], [c for c in cat_cols if c not in cond]),
}
for name, (n_, c_) in tests.items():
    m = -cross_val_score(make_pipe(n_, c_), df_ml[n_ + c_], y, cv=cv, scoring="neg_mean_absolute_error")
    print(f"{name:30s} MAE = {m.mean():.0f}")

In [ ]:
import joblib
from sklearn.model_selection import cross_val_predict

tarif = (df_ml.groupby("NTXINTANN")["montant_tnd"]
         .agg(n="size", mediane="median", minimum="min", maximum="max").round(0))
print(tarif[tarif["n"] >= 30].to_string())

os.makedirs("models", exist_ok=True)
num_B = [c for c in num_cols if c != "NTXINTANN"]
modele_A = make_pipe(num_cols, cat_cols).fit(df_ml[num_cols + cat_cols], y)
modele_B = make_pipe(num_B, cat_cols).fit(df_ml[num_B + cat_cols], y)
joblib.dump({"model": modele_A, "num": num_cols, "cat": cat_cols}, "models/montant_avec_taux.joblib")
joblib.dump({"model": modele_B, "num": num_B, "cat": cat_cols}, "models/montant_sans_taux.joblib")

ligne = df_ml.iloc[[0]]
print("\nExemple : réel =", round(float(ligne["montant_tnd"].iloc[0])),
      "| A =", round(float(modele_A.predict(ligne[num_cols + cat_cols])[0])),
      "| B =", round(float(modele_B.predict(ligne[num_B + cat_cols])[0])), "TND")

pred_cv = cross_val_predict(make_pipe(num_cols, cat_cols), df_ml[num_cols + cat_cols], y, cv=cv)
df_ml["ecart_tnd"] = y - pred_cv
cols = ["contract_id", "montant_tnd", "NECH", "NTXINTANN", "activite_contrat", "gouvernorat", "annee", "ecart_tnd"]
print("\nContrats les plus atypiques (écart prédit/réel) :")
print(df_ml.reindex(df_ml["ecart_tnd"].abs().sort_values(ascending=False).index)[cols].head(10).round(0).to_string(index=False))

In [ ]:
# Prediction des emplois : l'evaluation de production utilise un decoupage temporel.
# Le classifieur depoye et ses metriques sont produits par ce script reproductible.
df["emplois_cl"] = np.minimum(df["NEMPNV"], 2).astype(int)
print("Distribution cible :", df["emplois_cl"].value_counts().sort_index().to_dict())
os.makedirs("data", exist_ok=True)
df.to_pickle("data/contrats_ml.pkl")
from api.train_emplois import main as entrainer_emplois
entrainer_emplois()


In [ ]:
os.makedirs("data", exist_ok=True)
df.to_pickle("data/contrats_ml.pkl")
print(df.shape)

In [ ]:
import numpy as np, pandas as pd, os, joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

df = pd.read_pickle("data/contrats_ml.pkl")
os.makedirs("models", exist_ok=True)

In [ ]:
# Metriques de l'artefact de production, evalue sur les 12 derniers mois tenus a l'ecart.
bundle_emplois = joblib.load("models/emplois_classif.joblib")
print(bundle_emplois["evaluation"])


In [ ]:
t = (pd.crosstab(df["gouvernorat"], df["emplois_cl"], normalize="index")
       .round(3).rename(columns={0: "0", 1: "1", 2: "2+"}))
t["n"] = df["gouvernorat"].value_counts()
print(t.sort_values("2+", ascending=False).to_string())

print(pd.crosstab(df["annee"], df["emplois_cl"], normalize="index").round(3))

In [ ]:
import numpy as np, pandas as pd, os, joblib
df = pd.read_pickle("data/contrats_ml.pkl")
os.makedirs("models", exist_ok=True)

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

serie = (df.assign(mois_debut=pd.to_datetime(df["date_contrat"]).dt.to_period("M").dt.to_timestamp())
           .groupby("mois_debut").size().rename("nb_contrats"))
serie = serie.asfreq("MS", fill_value=0)
serie = serie.iloc[:-1]          # dernier mois (mai 2023) incomplet
print(serie.index.min(), serie.index.max(), len(serie))
print(serie.tail(15).to_string())

H = 12
train, test = serie.iloc[:-H], serie.iloc[-H:]
preds = {
    "Naïf (dernier mois)": pd.Series(train.iloc[-1], index=test.index),
    "Moyenne 6 derniers mois": pd.Series(train.iloc[-6:].mean(), index=test.index),
    "Saisonnier naïf (année-1)": pd.Series(train.iloc[-12:].to_numpy(), index=test.index),
}
preds["Lissage exponentiel amorti"] = ExponentialSmoothing(train, trend="add", damped_trend=True).fit().forecast(H)
preds["ARIMA(1,1,1)"] = SARIMAX(train, order=(1, 1, 1)).fit(disp=False).forecast(H)

for nom, p in preds.items():
    err = (test - p).abs()
    print(f"{nom:30s} MAE = {err.mean():6.1f} contrats/mois | erreur relative = {err.sum() / test.sum():.1%}")

plt.figure(figsize=(11, 4))
serie.plot(color="black", label="réel")
for nom, p in preds.items():
    p.plot(label=nom, alpha=0.8)
plt.legend(fontsize=8); plt.title("Contrats par mois : test sur les 12 derniers mois"); plt.show()

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

def prevoir(nom, hist, h):
    if nom == "Naïf (dernier mois)": return np.repeat(hist.iloc[-1], h)
    if nom == "Moyenne 3 mois": return np.repeat(hist.iloc[-3:].mean(), h)
    if nom == "Moyenne 6 mois": return np.repeat(hist.iloc[-6:].mean(), h)
    if nom == "Lissage amorti": return ExponentialSmoothing(hist, trend="add", damped_trend=True).fit().forecast(h).to_numpy()
    if nom == "ARIMA(1,1,1)": return SARIMAX(hist, order=(1, 1, 1)).fit(disp=False).forecast(h).to_numpy()

noms = ["Naïf (dernier mois)", "Moyenne 3 mois", "Moyenne 6 mois", "Lissage amorti", "ARIMA(1,1,1)"]
H = 3
erreurs = {n: [[] for _ in range(H)] for n in noms}
for o in range(len(serie) - 24, len(serie) - H + 1):          # 22 origines de prévision
    hist, futur = serie.iloc[:o], serie.iloc[o:o + H].to_numpy()
    for n in noms:
        p = prevoir(n, hist, H)
        for k in range(H):
            erreurs[n][k].append(abs(futur[k] - p[k]))

niveau = serie.iloc[-24:].mean()
tab = pd.DataFrame({n: [np.mean(erreurs[n][k]) for k in range(H)] for n in noms},
                   index=[f"MAE à +{k+1} mois" for k in range(H)]).T.round(0)
tab["erreur relative (moy.)"] = (tab.mean(axis=1) / niveau).map("{:.0%}".format)
print(f"Niveau moyen des 24 derniers mois : {niveau:.0f} contrats/mois\n")
print(tab.to_string())

res = SARIMAX(serie, order=(1, 1, 1)).fit(disp=False)
fc = res.get_forecast(H)
ic = fc.conf_int(alpha=0.2)
print("\nPrévision ARIMA des 3 prochains mois (intervalle 80 %) :")
print(pd.DataFrame({"prevision": fc.predicted_mean.round(0),
                    "bas": ic.iloc[:, 0].round(0), "haut": ic.iloc[:, 1].round(0)}).to_string())

In [ ]:
os.makedirs("data", exist_ok=True)
serie.to_csv("data/serie_mensuelle.csv")
prev = pd.DataFrame({"prevision": fc.predicted_mean.round(0),
                     "bas_80": ic.iloc[:, 0].round(0), "haut_80": ic.iloc[:, 1].round(0)})
prev.to_csv("data/prevision_3_mois.csv")
res.save("models/prevision_arima.pkl")
print(prev.to_string())

In [ ]:
import numpy as np, pandas as pd, os, joblib
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_pickle("data/contrats_ml.pkl")
os.makedirs("models", exist_ok=True)

In [ ]:
d = df[df["NECH"].notna()].copy()
d["log_montant"] = np.log1p(d["montant_tnd"])
d["emp_nv"] = d["NEMPNV"].clip(upper=5)
d["emp_ext"] = d["NEMPEXT"].clip(upper=5)
num_S = ["log_montant", "NECH", "NTXTEG", "emp_nv", "emp_ext"]
cat_S = ["activite_grp"]
prepS = ColumnTransformer([("num", StandardScaler(), num_S),
                           ("cat", OneHotEncoder(sparse_output=False), cat_S)])
XS = prepS.fit_transform(d[num_S + cat_S])

rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(XS)
    rows.append((k, km.inertia_, silhouette_score(XS, km.labels_, sample_size=3000, random_state=42)))
print(pd.DataFrame(rows, columns=["k", "inertie", "silhouette"]).round(3).to_string(index=False))

K = 4      # à ajuster d'après le tableau ci-dessus
km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(XS)
d["segment"] = km.labels_
g = d.groupby("segment")
profil = g.agg(n=("montant_tnd", "size"), montant_med=("montant_tnd", "median"),
               duree_med=("NECH", "median"), teg_moy=("NTXTEG", "mean"),
               emplois_nv=("NEMPNV", "mean"), emplois_ext=("NEMPEXT", "mean")).round(1)
profil["part_%"] = (profil["n"] / len(d) * 100).round(1)
profil["activite_dominante"] = g["activite_grp"].agg(lambda s: s.value_counts().index[0])
profil["part_activite_%"] = g["activite_grp"].agg(lambda s: s.value_counts(normalize=True).iloc[0] * 100).round(0)
print()
print(profil.to_string())

In [ ]:
def nommer(r):
    if r["montant_med"] <= 5000: return "Micro-crédits (environ 3 000 TND)"
    if r["emplois_ext"] > 3: return "Gros crédits à fort impact emploi"
    if r["duree_med"] <= 12: return "Crédits moyens courts"
    return "Crédits moyens standards"

noms = profil.apply(nommer, axis=1).to_dict()
d["segment_nom"] = d["segment"].map(noms)
print(noms, "\n")
print(pd.crosstab(d["annee"], d["segment_nom"], normalize="index").round(2).to_string())

os.makedirs("data", exist_ok=True)
joblib.dump({"prep": prepS, "km": km, "num": num_S, "cat": cat_S, "noms": noms}, "models/segmentation.joblib")
profil.assign(nom=profil.index.map(noms)).to_csv("data/profil_segments.csv")
d[["contract_id", "segment", "segment_nom"]].to_csv("data/segments_contrats.csv", index=False)
print("\nfichiers sauvegardés")